# Step 1.3 Create Table HRS Variable Candidates
This notebook creates the HRS Variable Candidaes.  

I looks for term that are potential variables which must be reviewed and approved later in the workflow.

# Setup Global Variables

In [0]:
from pyspark.sql import functions as F, types as T

CATALOG = 'dev_catalog'
SCHEMA = 'hrs_agent_metadata'
CODEBOOK_VERSION = '1992-2022v1'

# f-string to construct a single, dot-separated strings.
PAGES = f'{CATALOG}.{SCHEMA}.hrs_document_pages'
CANDIDATES = f'{CATALOG}.{SCHEMA}.hrs_variable_candidates'

# Create the Candidates table if it does not exist.

In [0]:
#from pyspark.sql import functions as F
#from pyspark.sql.window import Window
#from delta.tables import DeltaTable

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
catalog_schema = "dev_catalog.hrs_agent_metadata"
candidates_table = f"{catalog_schema}.hrs_variable_candidates"

# Define the DDL statement
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {candidates_table} (
    document_id      STRING NOT NULL    COMMENT 'Source document identifier',
    source_page      INT NOT NULL       COMMENT 'Page where the candidate was found',
    candidate_token  STRING NOT NULL    COMMENT 'Extracted candidate, e.g. R1BMI',
    context_excerpt  STRING             COMMENT 'Text surrounding the candidate',
    codebook_version STRING NOT NULL    COMMENT 'Codebook version',
    created_at       TIMESTAMP          COMMENT 'Extraction timestamp'
    )
    USING DELTA
""")

print("Table 'candidate_rows' validated/created successfully.")

# Confirm the pages table.

This tells us whether the Codebook extraction has already run successfully and what columns are available. The automated extraction notebook needs that information.

In [0]:
pages_table = "dev_catalog.hrs_agent_metadata.hrs_document_pages"

print("Table exists:", spark.catalog.tableExists(pages_table))

if spark.catalog.tableExists(pages_table):
    display(spark.table(pages_table).limit(5))
    print("Page count:", spark.table(pages_table).count())
    print("Columns:", spark.table(pages_table).columns)

# Discover "candidate" variable mentions for review

### Parse the Pages table and look for potential variable names

* This is a broad "candidate" finder, not a definitive parser. Review results before loading anything into the final metadata tables.

* Candidates are potential HRS variable names identified during Codebook text extraction, retained with source-page references and context excerpts for review and verification before approval.

The code that follows is a bit more complex.  It requires analysis to understand the data extraction approach.

In [0]:
5# Import the Regular Expression module
import re

# Read the pages table and load a PySpark DataFrame
pages_df = spark.table(PAGES)

# Create a Python dataframe containing the minimum columns required for the rest of your script to function: 'page_number' and 'page_text'
#     And verify (issubset) the columns exist.  
#     This just test for the existence of the columns, not the values in the columns.
#     If not columns are found raise the ERROR and Stop the script.
required_cols = {'page_number', 'page_text'}
if not required_cols.issubset(set(pages_df.columns)): 
    raise ValueError(f'{PAGES} must contain columns {sorted(required_cols)}; update the column mapping.') 

# Check codebook version is in the pages table. # If not, remove those with different codebook versions.
if 'codebook_version' in pages_df.columns:
    pages_df = pages_df.filter(F.col('codebook_version') == CODEBOOK_VERSION)

# Scan the text of every page to find uppercase keywords (tokens), filters out junk words, and extracts a text excerpt surrounding each match to create a collection of text-mining features.
token_re = re.compile(r'(?<![A-Za-z0-9_])([A-Z][A-Z0-9_]{1,19})(?![A-Za-z0-9_])')

ignore = {'THE', 'AND', 'FOR', 'FROM', 'WITH', 'WAVE', 'PAGE', 'TABLE', 'NOTE'}

# Build the column list to select. 
# If document_id exists in your schema, it includes it; otherwise only grab page metadata and text.

# Initialize and empty list
candidate_rows = []

#  list of column names that you want to select or process later.
cols = ['page_number', 'page_text'] + (['document_id'] if 'document_id' in pages_df.columns else [])

#processes a text DataFrame page by page to extract unique uppercase abbreviations/tokens 
for r in pages_df.select(*cols).filter(F.col('page_text').isNotNull()).toLocalIterator():
    text = r['page_text'] or ''
    seen = set()
    # Regex Token Matching & Deduplication
    for m in token_re.finditer(text):
        token = m.group(1)
        if token in ignore or token in seen:
            continue
        seen.add(token)
        # Append document_id, page_number, token, text excerpt, and codebook version
        candidate_rows.append((str(r['document_id']) if 'document_id' in r.asDict() and r['document_id'] else 'rand_hrs_codebook', int(r['page_number']), token, text[max(0,m.start()-160):min(len(text),m.end()+220)], CODEBOOK_VERSION))

display(candidate_rows)

# Convert Python List to a Spark Data Frame

In [0]:
# Create the Candidate Schema layout structure
candidate_schema = T.StructType([
    T.StructField('document_id',T.StringType(),False), 
    T.StructField('source_page',T.IntegerType(),False), 
    T.StructField('candidate_token',T.StringType(),False), 
    T.StructField('context_excerpt',T.StringType(),True), 
    T.StructField('codebook_version',T.StringType(),False)
])

# Take the raw Python list of extracted text tokens (candidate_rows) 
#      and convert it back into a structured Spark DataFrame named candidate_df 
#      using a predefined layout blueprint (candidate_schema). 
candidate_df = spark.createDataFrame(candidate_rows, candidate_schema) \
    .withColumn('created_at', F.current_timestamp())


# Load Candidates Dataframe to Delta Table

In [0]:
# Write to Delta Lake and create the CANDIDATES table: hrs_variable_candidates
#candidate_df.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable(CANDIDATES)
candidate_df.write.format('delta').mode('append').saveAsTable(CANDIDATES)

print(f'Wrote {len(candidate_rows):,} candidate mentions to {CANDIDATES}')
# display(candidate_df.orderBy('candidate_token','source_page').limit(300))

## Before Step 1.4

- Verify every loaded definition, statistic, value label, and coverage record against the original Codebook.
- Confirm duplicate-key groups and orphan variable IDs are zero.
- Confirm source page references are accurate.
- Do not infer cohort coverage where it is not documented.

This notebook provides controlled DML, not a fully automatic Codebook parser. The next refinement should use representative page text from the actual Codebook to improve automated extraction.